# 05 — Pipeline B: YOLO Single-class + ResNet50 End-to-End

**Workflow:** Overlap filter → YOLO single-class → ResNet50 classification → Count validation → Karyotype assembly → Autoencoder anomaly score

Requires model weights in place (see `models/README.md`).


In [ ]:
import os
FLAG = "/content/.deps_ok_karyotype"
if not os.path.exists(FLAG):
    !pip -q install --no-cache-dir --upgrade --force-reinstall \
        "numpy==2.0.2" "pandas==2.2.2" "opencv-python==4.10.0.84" \
        "ultralytics==8.*" "torch" "torchvision" "tqdm==4.*" "matplotlib==3.*"
    open(FLAG,"w").write("ok")
    raise SystemExit("✅ Restart runtime, then re-run.")
print("✅ Dependencies OK")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, math, shutil
from pathlib import Path
import numpy as np, cv2, pandas as pd
from tqdm import tqdm
import torch
from ultralytics import YOLO
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input

# ── CONFIG ────────────────────────────────────────────────────────────
ROOT     = "/content/drive/MyDrive/54816"
OUT_DIR  = f"{ROOT}/e2e_run_pipeline_B"
os.makedirs(OUT_DIR, exist_ok=True)

OVERLAP_W   = "/content/drive/MyDrive/Modelos/overlap_modelo_entrenado2.pt"
YOLO1_W     = f"{ROOT}/weight/best_single_chromosomes.pt"
RESNET_W    = "/content/drive/MyDrive/ResNet50_cromosomas_model.keras"
AE_KERAS    = f"{ROOT}/weights_autoencoder/autoencoder_model.keras"

TARGET_DIR  = f"{ROOT}/24_chromosomes_object/JEPG"

OVERLAP_LABEL = "overlapped"
OVERLAP_CONF  = 0.25
YOLO_CONF     = 0.25
YOLO_IOU      = 0.70
RESNET_SIZE   = (224,224)
AE_IMG_SIZE   = (128,128)

KARYO_ORDER = [
    "A1","A2","A3","B4","B5",
    "C6","C7","C8","C9","C10","C11","C12",
    "D13","D14","D15","E16","E17","E18",
    "F19","F20","G21","G22","X","Y"
]

DEVICE = 0 if torch.cuda.is_available() else "cpu"

def exists(p): print("✅" if Path(p).exists() else "❌", p)
for p in [OVERLAP_W, YOLO1_W, RESNET_W, TARGET_DIR]: exists(p)
print("AE:", "✅" if Path(AE_KERAS).exists() else "⚠️ optional", AE_KERAS)


In [ ]:
# ── Utilities (shared with Pipeline A) ───────────────────────────────
IMG_EXTS={".jpg",".jpeg",".png",".bmp",".tif",".tiff"}
def list_images(d):
    files=[]
    for e in IMG_EXTS: files+=list(Path(d).rglob(f"*{e}")); files+=list(Path(d).rglob(f"*{e.upper()}"))
    return sorted(set(files))
def imread_rgb(p):
    bgr=cv2.imread(str(p),cv2.IMREAD_COLOR)
    return cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB) if bgr is not None else None
def crop_bbox(img,x1,y1,x2,y2,pad=0):
    h,w=img.shape[:2]
    x1=max(0,int(x1)-pad); y1=max(0,int(y1)-pad)
    x2=min(w,int(x2)+pad); y2=min(h,int(y2)+pad)
    return img[y1:y2,x1:x2].copy() if x2>x1 and y2>y1 else None
def numerical_alert(counts):
    exp={k:2 for k in KARYO_ORDER if k not in ("X","Y")}
    exp["X"]=1 if counts.get("Y",0)>0 else 2
    exp["Y"]=1 if counts.get("Y",0)>0 else 0
    return {k:{"obs":int(counts.get(k,0)),"exp":int(e)}
            for k,e in exp.items() if int(counts.get(k,0))!=int(e)}
def build_karyotype(crops_by_label,out_path,cell=(220,220),cols=4,pad=10):
    cw,ch=cell; order=KARYO_ORDER; n=len(order); rows=math.ceil(n/cols)
    canvas=np.ones((rows*ch+(rows+1)*pad,cols*cw+(cols+1)*pad,3),np.uint8)*255
    for idx,lbl in enumerate(order):
        r,c=idx//cols,idx%cols; y0=pad+r*(ch+pad); x0=pad+c*(cw+pad)
        imgs=crops_by_label.get(lbl,[]); cell_img=np.ones((ch,cw,3),np.uint8)*255
        if imgs:
            half=(ch-10)//2
            def fit(img,th): h,w=img.shape[:2]; s=min(th/h,cw/w); return cv2.resize(img,(max(1,int(w*s)),max(1,int(h*s))),cv2.INTER_AREA)
            if len(imgs)==1:
                t=fit(imgs[0],ch); oy=(ch-t.shape[0])//2; ox=(cw-t.shape[1])//2; cell_img[oy:oy+t.shape[0],ox:ox+t.shape[1]]=t
            else:
                for img2,y_off in zip(imgs[:2],[0,half+10]):
                    t=fit(img2,half); ox=(cw-t.shape[1])//2; cell_img[y_off:y_off+t.shape[0],ox:ox+t.shape[1]]=t
        canvas[y0:y0+ch,x0:x0+cw]=cell_img
    cv2.imwrite(str(out_path),cv2.cvtColor(canvas,cv2.COLOR_RGB2BGR)); return out_path


In [ ]:
# ── Stage 1: Overlap gate ─────────────────────────────────────────────
overlap_model = YOLO(OVERLAP_W)
names=overlap_model.names; id2n=names if isinstance(names,dict) else {i:n for i,n in enumerate(names)}
n2id={str(v).strip().lower():int(k) for k,v in id2n.items()}; ov_id=n2id[OVERLAP_LABEL.lower()]

all_imgs=list_images(TARGET_DIR); kept=[]; discarded=[]
for p in tqdm(all_imgs,desc="Overlap gate"):
    r=overlap_model(str(p),verbose=False,device=DEVICE,conf=OVERLAP_CONF)[0]
    ids=[int(c) for c in r.boxes.cls] if r.boxes and len(r.boxes) else []
    (discarded if ov_id in ids else kept).append(p)
print(f"Kept {len(kept)}/{len(all_imgs)} images.")


In [ ]:
# ── Stage 2: Load YOLO single + ResNet50 ─────────────────────────────
yolo1       = YOLO(YOLO1_W)
resnet_model= load_model(RESNET_W)
print("✅ Models loaded")

# Load class names (use saved mapping from NB02 if available)
classes_json=os.path.join(ROOT,"e2e_outputs_run","resnet_classes.json")
if os.path.isfile(classes_json):
    resnet_classes=json.load(open(classes_json))
    print("[INFO] ResNet classes from file:", resnet_classes)
else:
    resnet_classes=KARYO_ORDER
    print("[INFO] Using default KARYO_ORDER as ResNet class names")

def classify_crop(crop_rgb):
    x=cv2.resize(crop_rgb,RESNET_SIZE).astype(np.float32)
    x=np.expand_dims(x,0); x=preprocess_input(x)
    probs=resnet_model.predict(x,verbose=0)[0]
    idx=int(np.argmax(probs))
    return resnet_classes[idx], float(probs[idx])


In [ ]:
# ── Stages 3-5: Detect → classify → count → assemble ────────────────
karyo_dir=Path(OUT_DIR)/"karyotypes_B"; karyo_dir.mkdir(exist_ok=True)
results=[]
for img_path in tqdm(kept,desc="Pipeline B"):
    img=imread_rgb(img_path)
    if img is None: continue
    r=yolo1(str(img_path),verbose=False,device=DEVICE,conf=YOLO_CONF,iou=YOLO_IOU)[0]
    counts={}; crops_by_label={}
    if r.boxes and len(r.boxes):
        for box in r.boxes:
            x1,y1,x2,y2=box.xyxy[0].tolist()
            crop=crop_bbox(img,x1,y1,x2,y2)
            if crop is None: continue
            lbl,conf=classify_crop(crop)
            counts[lbl]=counts.get(lbl,0)+1
            crops_by_label.setdefault(lbl,[]).append(crop)
    alerts=numerical_alert(counts)
    karyo_path=karyo_dir/f"{img_path.stem}_B.png"
    build_karyotype(crops_by_label,karyo_path)
    results.append({"image":img_path.name,"counts":counts,
                    "alerts":alerts,"karyotype":str(karyo_path),"ae_scores":None})
print(f"Processed {len(results)} images.")


In [ ]:
# ── Stage 6 (optional): Autoencoder ──────────────────────────────────
if Path(AE_KERAS).exists():
    ae=load_model(AE_KERAS)
    def ae_err(c): x=cv2.resize(c,AE_IMG_SIZE).astype(np.float32)/255.0; xb=x[None]; return float(np.mean((x-ae.predict(xb,verbose=0)[0])**2))
    all_crops=[]
    for entry in results:
        img=imread_rgb(Path(TARGET_DIR)/entry["image"])
        if img is None: continue
        r=yolo1(str(Path(TARGET_DIR)/entry["image"]),verbose=False,device=DEVICE,conf=YOLO_CONF,iou=YOLO_IOU)[0]
        if r.boxes and len(r.boxes):
            for box in r.boxes:
                x1,y1,x2,y2=box.xyxy[0].tolist(); c=crop_bbox(img,x1,y1,x2,y2)
                if c is not None: all_crops.append(c)
    thr=float(np.percentile([ae_err(c) for c in all_crops],95)) if all_crops else 0.02
    print(f"AE threshold (p95): {thr:.6f}")
    for entry in results:
        img=imread_rgb(Path(TARGET_DIR)/entry["image"])
        if img is None: continue
        r=yolo1(str(Path(TARGET_DIR)/entry["image"]),verbose=False,device=DEVICE,conf=YOLO_CONF,iou=YOLO_IOU)[0]
        ae_flags={}
        if r.boxes and len(r.boxes):
            for box in r.boxes:
                x1,y1,x2,y2=box.xyxy[0].tolist(); c=crop_bbox(img,x1,y1,x2,y2)
                if c is None: continue
                lbl,_=classify_crop(c); err=ae_err(c)
                if err>ae_flags.get(lbl,{}).get("max_err",0): ae_flags[lbl]={"max_err":err,"anomaly":err>thr}
        entry["ae_scores"]=ae_flags
    print("✅ AE scoring complete")
else:
    print("⚠️ AE model not found — skipping.")


In [ ]:
out_path=os.path.join(OUT_DIR,"pipeline_b_results.json")
with open(out_path,"w") as f: json.dump(results,f,indent=2,ensure_ascii=False,default=str)
alerts_count=sum(1 for r in results if r["alerts"])
print("Results saved to:", out_path)
print(f"Images with numerical alerts: {alerts_count}/{len(results)}")
